# Step 5 — Summarization Prompts

**Project:** Prompt Engineering for Clothing Review Analysis  
**Dataset:** Women's Clothing E-Commerce Reviews (`data/reviews.csv`)  
**Goal:** Test and compare single-review and multi-review summarization prompts across naive and improved versions.

This notebook generates ready-to-test prompts for:
1. **Single-review summarization:** Condensing long, detailed customer feedback into a concise, actionable sentence.
2. **Multi-review synthesis:** Aggregating multiple reviews for a single product category into structured, frequency-backed insights.

## 0. Setup and Data Loading

We load the dataset using a resilient path check so it works whether Jupyter is run from the repository root or the `prompts/` subfolder.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 120)

# Resilient file path
DATA_PATH = (
    Path("data/reviews.csv")
    if Path("data/reviews.csv").exists()
    else Path("..") / "data" / "reviews.csv"
)

df_raw = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head(3)

Loaded: C:\GamageRecruiters-DataScienceIntern\Month_02\prompt-engineering-task\data\reviews.csv
Raw dataset shape: (23486, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap...",3,0,0,General,Dresses,Dresses


## 1. Clean Review Text

We clean the text using the exact findings from our EDA:
1. Drop rows with missing `Review Text` (845 blank rows).
2. Normalize literal Windows line breaks (`\r\n`) into single spaces.

In [2]:
df = df_raw.copy()

# 1. Drop missing review text
df = df.dropna(subset=["Review Text"])

# 2. Normalize whitespace and remove \r\n linebreaks
def normalize_review_text(text):
    text = str(text)
    text = text.replace("\r\n", " ")
    text = text.strip()
    return text

df["Review Text"] = df["Review Text"].map(normalize_review_text)
df = df[df["Review Text"].str.len() > 0].copy()

print(f"Usable clean reviews: {len(df):,}")

Usable clean reviews: 22,641


## 2. Sampling Strategy

Summarization tasks require two distinct data views:
- **Long Reviews (Single-review test):** Selecting the longest reviews in the catalog tests whether an LLM can condense detailed, multi-topic prose without losing the main point.
- **Category Grouping (Multi-review test):** Sampling reviews from a single class (e.g., `Dresses`) tests whether an LLM can identify recurring cross-customer patterns.

In [3]:
def pick_long_reviews(df, n=3, seed=42):
    """Select n reviews with the longest character count across diverse classes."""
    temp_df = df.copy()
    temp_df["char_length"] = temp_df["Review Text"].str.len()
    sorted_df = temp_df.sort_values(by="char_length", ascending=False)
    
    selected_rows = []
    seen_classes = set()
    
    for _, row in sorted_df.iterrows():
        cls = row["Class Name"]
        if cls not in seen_classes and pd.notna(cls):
            selected_rows.append(row)
            seen_classes.add(cls)
        if len(selected_rows) == n:
            break
            
    return pd.DataFrame(selected_rows)

def pick_reviews_by_class(df, class_name="Dresses", n=8, seed=42):
    """Sample n reviews from a specific product class."""
    class_df = df[df["Class Name"] == class_name]
    return class_df.sample(n=n, random_state=seed)

# Generate samples
long_reviews_df = pick_long_reviews(df, n=3)
category_reviews_df = pick_reviews_by_class(df, class_name="Dresses", n=8)

print("Top Long Reviews Selected:")
display(long_reviews_df[["Clothing ID", "Class Name", "Rating", "char_length"]])

print("\nCategory Sample (Dresses) Selected:")
display(category_reviews_df[["Clothing ID", "Rating", "Recommended IND", "Review Text"]].head(3))

Top Long Reviews Selected:


,Clothing ID,Class Name,Rating,char_length
17597,824,Blouses,5,508
19304,1103,Dresses,1,508
22010,909,Fine gauge,1,506



Category Sample (Dresses) Selected:


,Clothing ID,Rating,Recommended IND,Review Text
17216,1072,5,1,"This dress is stunning. i have been stopped in the street by people asking where the dress is from. my only complaint is that it was very wrinkled when i bought it. initially i thought it was supposed to be that way, but after seeing the stock photos i realized it needed to be ironed. it took a long time to iron all of the wrinkles out, but it is totally worth it."
15797,1094,2,0,"Review title says almost all. add to that that it is age appropriate, unlike other kedia dresses, for the very young. just too cute for words. i have all of the other tk peasant dresses and although youthful they are appropriate for anyone young of heart. this however, on an older woman, looks ridiculous. i would have liked it for those 9 months but even then it would cover quadruplets!"
10833,1110,5,1,"I had only planned to check out this dress at the store but ended up buying it then and there! not a deal breaker but there were extra threads hanging from some parts of the dress. it made me a bit concern that the stitching would unravel during wear. so far, so good! i wish they had paid more attention to little details like that."


## 3. Formatting Prompt Data

We convert our dataframes into clean, human-readable strings to append directly to our prompts.

In [4]:
# Select the single longest review for the single-review experiment
target_single_review = long_reviews_df.iloc[0]

single_review_text = (
    f"Clothing Item: {target_single_review['Class Name']} (ID: {target_single_review['Clothing ID']})\n"
    f"Rating: {target_single_review['Rating']} out of 5\n"
    f"Customer Review: {target_single_review['Review Text']}"
)

# Format the 8 multi-review samples
multi_lines = [f"Here are 8 customer reviews for category 'Dresses':\n"]
for idx, (_, row) in enumerate(category_reviews_df.iterrows(), start=1):
    rec_status = "Recommended" if row["Recommended IND"] == 1 else "Not Recommended"
    multi_lines.append(f"--- Review {idx} ---")
    multi_lines.append(f"Rating: {row['Rating']}/5 | Status: {rec_status}")
    multi_lines.append(f"Review: {row['Review Text']}\n")

multi_reviews_text = "\n".join(multi_lines).strip()

## 4. Prompt Engineering: Single-Review Prompts

- `single_v1_naive`: Generic command with zero output constraints.
- `single_v2_improved`: Assigns a specific role, requires a one-sentence constraint, mandates preserving the primary praise/complaint, and strictly forbids hallucinations.

In [5]:
# Version 1: Naive
single_v1_naive = "Summarize this review."

# Version 2: Improved
single_v2_improved = """
You are an e-commerce customer experience analyst.

Task:
Summarize the customer review below into exactly ONE sentence.

Constraints:
1. Preserve the core complaint and/or primary praise mentioned by the shopper.
2. Do not exceed 25 words.
3. Rely strictly on facts mentioned in the review. Do not add assumptions or interpretations.
4. Return only the summary sentence without any conversational filler.
""".strip()

## 5. Prompt Engineering: Multi-Review Prompts

- `multi_v1_naive`: Basic request prone to ungrounded generalizations.
- `multi_v2_improved`: Requests an executive synthesis with exactly 3 bullet points, quantitative evidence counts (e.g., 'X of 8 reviews'), and direct supporting quotes.

In [6]:
# Version 1: Naive
multi_v1_naive = "Summarize these reviews."

# Version 2: Improved
multi_v2_improved = """
You are a merchandising insights analyst.

Task:
Analyze the 8 customer reviews for the 'Dresses' category below and synthesize the recurring patterns into an executive briefing for a category manager.

Format and Content Constraints:
1. Output exactly 3 bullet points, each focusing on a distinct theme (e.g., Sizing/Fit, Fabric Quality, Versatility).
2. Each bullet point MUST state the exact number of supporting reviews (e.g., "[Theme]: X of 8 reviews indicate...").
3. Include one short direct quote (3-6 words) as evidence within each bullet.
4. Do not include introductory or concluding conversational text.
""".strip()

## 6. Generate Copy-Ready Prompts

Run the cells below to print the prompts with attached data. Copy the text blocks directly into Cursor Chat, ChatGPT, or Claude.

In [7]:
print("=" * 72)
print("COPY FROM HERE — single_v1 (naive)")
print("=" * 72)
print(f"{single_v1_naive}\n\n{single_review_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — single_v1 (naive)
Summarize this review.

Clothing Item: Blouses (ID: 824)
Rating: 5 out of 5
Customer Review: I adore this blouse. the colors are vibrant (see my photo below). this is one of my favorite purchases from retailer. the top is light weight. true to size. i ordered a petite small and am 5 feet tall 120 lbs. and curvy. i left it untucked and loose like in the photo and it was very flattering. i disagree about it being frumpy. i wore it with kelly green retailer brand slacks paired with the retailer yellow sweater coat with white piping, and the retailer moss suede cross bag and neutral color (nude) fl
COPY TO HERE


In [8]:
print("=" * 72)
print("COPY FROM HERE — single_v2 (improved)")
print("=" * 72)
print(f"{single_v2_improved}\n\n{single_review_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — single_v2 (improved)
You are an e-commerce customer experience analyst.

Task:
Summarize the customer review below into exactly ONE sentence.

Constraints:
1. Preserve the core complaint and/or primary praise mentioned by the shopper.
2. Do not exceed 25 words.
3. Rely strictly on facts mentioned in the review. Do not add assumptions or interpretations.
4. Return only the summary sentence without any conversational filler.

Clothing Item: Blouses (ID: 824)
Rating: 5 out of 5
Customer Review: I adore this blouse. the colors are vibrant (see my photo below). this is one of my favorite purchases from retailer. the top is light weight. true to size. i ordered a petite small and am 5 feet tall 120 lbs. and curvy. i left it untucked and loose like in the photo and it was very flattering. i disagree about it being frumpy. i wore it with kelly green retailer brand slacks paired with the retailer yellow sweater coat with white piping, and the retailer moss suede cross bag and n

In [9]:
print("=" * 72)
print("COPY FROM HERE — multi_v1 (naive)")
print("=" * 72)
print(f"{multi_v1_naive}\n\n{multi_reviews_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — multi_v1 (naive)
Summarize these reviews.

Here are 8 customer reviews for category 'Dresses':

--- Review 1 ---
Rating: 5/5 | Status: Recommended
Review: This dress is stunning. i have been stopped in the street by people asking where the dress is from. my only complaint is that it was very wrinkled when i bought it. initially i thought it was supposed to be that way, but after seeing the stock photos i realized it needed to be ironed. it took a long time to iron all of the wrinkles out, but it is totally worth it.

--- Review 2 ---
Rating: 2/5 | Status: Not Recommended
Review: Review title says almost all. add to that that it is age appropriate, unlike other kedia dresses, for the very young. just too cute for words. i have all of the other tk peasant dresses and although youthful they are appropriate for anyone young of heart. this however, on an older woman, looks ridiculous. i would have liked it for those 9 months but even then it would cover quadruplets!

--- Re

In [10]:
print("=" * 72)
print("COPY FROM HERE — multi_v2 (improved)")
print("=" * 72)
print(f"{multi_v2_improved}\n\n{multi_reviews_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — multi_v2 (improved)
You are a merchandising insights analyst.

Task:
Analyze the 8 customer reviews for the 'Dresses' category below and synthesize the recurring patterns into an executive briefing for a category manager.

Format and Content Constraints:
1. Output exactly 3 bullet points, each focusing on a distinct theme (e.g., Sizing/Fit, Fabric Quality, Versatility).
2. Each bullet point MUST state the exact number of supporting reviews (e.g., "[Theme]: X of 8 reviews indicate...").
3. Include one short direct quote (3-6 words) as evidence within each bullet.
4. Do not include introductory or concluding conversational text.

Here are 8 customer reviews for category 'Dresses':

--- Review 1 ---
Rating: 5/5 | Status: Recommended
Review: This dress is stunning. i have been stopped in the street by people asking where the dress is from. my only complaint is that it was very wrinkled when i bought it. initially i thought it was supposed to be that way, but after seeing t

## Output — single_v1 (paste here)

* **Rating:** 5/5 stars
* **Fit & Sizing:** Fits true to size and flatters curvy petite frames (reviewer is 5'0", 120 lbs, wearing Petite Small).
* **Fabric & Look:** Features vibrant colors and a lightweight feel; wears loose and untucked without looking frumpy.
* **Styling:** Highly versatile—reviewer paired it with kelly green slacks, a yellow sweater coat, a moss green crossbody bag, and nude flats.

## Output — single_v2 (paste here)

The shopper loved the true-to-size, lightweight blouse, praising its vibrant colors, flattering untucked fit, and overall look.

## Output — multi_v1 (paste here)

Customer sentiment is overwhelmingly positive, with reviewers praising the stylish, versatile bohemian look, though noting minor quality-control flaws and a very voluminous cut.

**Key Highlights**

* **Style & Compliments:** Customers love the aesthetic, calling it stunning, bohemian, and versatile enough to dress up or down. Multiple buyers noted receiving compliments when wearing it.
* **Fit & Sizing:** The cut is loose and swingy. While one reviewer appreciated that it didn't feel like a tent after sizing down from Small to XS, another found the volume overwhelming and excessively youthful, comparing it to oversized maternity wear.
* **Finishing Flaws:** Minor quality control issues were reported, including heavy initial wrinkling that required tedious ironing and loose, stray threads out of the box.

Overall, buyers feel the dress justifies its price point once ironed and properly sized, provided you like a breezy, voluminous silhouette.

## Output — multi_v2 (paste here)

* **Sizing & Fit**: 2 of 8 reviews indicate that voluminous or swing silhouettes risk feeling excessively oversized, requiring customers to size down so the garment does not look like "it would cover quadruplets."
* **Perceived Value**: 2 of 8 reviews indicate strong customer satisfaction that justifies higher price points and garment care effort, noting the purchase is "it is totally worth it."
* **Aesthetic Appeal**: 2 of 8 reviews indicate that eye-catching designs generate immediate positive reception and public compliments, with customers highlighting that "this dress is stunning."

## Comparison notes

### 1. Single-Review Summarization (v1 vs. v2)
- **What was vague/unclear in v1:** 
  - Instead of condensing the text, the model decomposed the review into an itemized multi-category profile (Rating, Fit, Fabric, Styling).
  - It included low-priority extraneous details (such as listing every specific clothing accessory the reviewer paired with the top) rather than extracting an executive summary.
- **What specifically improved in v2:** 
  - The model compressed the feedback into a single 18-word sentence, well under the 25-word cap.
  - It preserved only the core praise (true to size, lightweight, vibrant colors, flattering untucked fit) without conversational filler.
- **Key prompt drivers (why it changed):** 
  - Specifying strict structural boundaries ("exactly ONE sentence", "do not exceed 25 words") forced the LLM to discard secondary styling details and isolate high-signal adjectives.

### 2. Multi-Review Synthesis (v1 vs. v2)
- **What was vague/unclear in v1:** 
  - Relied on ungrounded quantifiers ("multiple buyers", "one reviewer", "another") without specifying actual sample distribution.
  - Included conversational padding ("Overall, buyers feel...") and loosely paraphrased negative sentiment without verifiable evidence.
- **What specifically improved in v2:** 
  - The output strictly matched the required executive briefing format: exactly 3 bullets, each tracking a distinct category theme.
  - Every theme explicitly included sample frequencies ("2 of 8 reviews") and verbatim direct quotes ("it would cover quadruplets", "it is totally worth it", "this dress is stunning") for grounding.
- **Key prompt drivers (why it changed):** 
  - Enforcing the persona of a merchandising insights analyst and mandating quantifiable metrics ("X of 8 reviews") eliminated generic generalizations and anchored findings directly in empirical evidence.